# ML-04 — Search Intelligence Data Contract

**Lane:** Structured Content Archetype Clustering  
**Development month:** March 2026  
**Temporary leakage-demo outcome month:** April 2026  
**Sealed test month:** June 2026

This notebook defines and verifies the slice of the FlyRank warehouse used for my lane. The main capstone remains an **unsupervised clustering task**. A temporary April proxy is created only for the required leakage demonstration; it is not the final capstone target.

Public-safety rules followed here:

- no client names, URLs, titles, domains, or raw search queries;
- pseudonymized IDs are used only for grouping and joining;
- no Hugging Face token is printed or stored in a code cell;
- the final June 2026 month is not used for feature or label development.


## 0. Setup and warehouse connection

Before running:

1. Request access to `FlyRank/internship-warehouse` on Hugging Face.
2. Create a plain **Read** token.
3. In Google Colab, open the key/Secrets panel and add a secret named `HF_TOKEN`.
4. Turn on notebook access for the secret.
5. Run this notebook from top to bottom.

The warehouse daily fact has one row per report date, pseudonymized client, and pseudonymized content item. March is used for development because it is a mid-panel month. June is the final month and stays sealed.


In [ ]:
# Install only what this notebook needs.
%pip -q install duckdb scikit-learn pandas

import os
import duckdb
import pandas as pd
import numpy as np

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Read the token from Colab Secrets. Never paste or print the token.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Add it in Colab Secrets or as an environment variable."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{safe_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"
APRIL = f"{BASE}/fact_content_daily_performance/month=2026-04/*.parquet"

# Display schema only. This helps catch a renamed warehouse column early.
schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{MARCH}')").df()
display(schema)


## 1. Contract in plain words

### 1. What one row means

The raw warehouse table is daily. After aggregation, **one feature-frame row means one pseudonymized content page for one client, summarized over March 2026**.

The combination of `client_id` and `content_id` identifies the page inside the warehouse. These IDs are context fields only and are never model features.

### 2. Tables used

- `fact_content_daily_performance`, March 2026 partition — feature window.
- `fact_content_daily_performance`, April 2026 partition — temporary future proxy used only for the leakage lesson.

I do not use the `_sample` table because it is the final month, June 2026.

### 3. Time window

- **Features:** 2026-03-01 through 2026-03-31.
- **Temporary proxy:** April 2026 performance.
- **Sealed month:** June 2026, not touched in this notebook.

### 4. What the lane produces

The actual capstone output is a cluster assignment that groups pages with similar performance patterns. There is no supervised target for the clustering model.

For the required leakage experiment only, I create a temporary binary proxy: whether a page's April CTR is above the median April CTR among eligible pages. This proxy is used to demonstrate why label-derived columns must never enter the feature set.

### 5. Deliberate exclusion

I deliberately exclude:

- June 2026 data, because it is the sealed final month.
- IDs as features, because they are pseudonymous join keys rather than behavioral signals.
- raw or future April measures from the honest feature set, because they are not knowable at the end of March.
- product flags or scores, because they may contain decision logic or label information.


## 2. Fields: feature / label / context / excluded

### Features — exactly five

1. `log_impressions` — log-transformed March GSC impressions.
2. `ctr_pct` — March clicks divided by March impressions, expressed as a percentage.
3. `avg_position` — March impression-weighted average search position.
4. `active_days` — number of March days with at least one impression.
5. `engagement_rate_pct` — March GA4 engaged sessions divided by sessions, using only days where `ga4_data_available IS TRUE`.

### Temporary label/proxy

- `future_high_ctr` — 1 when April CTR is above the eligible-page median, otherwise 0.
- It exists only for the leakage demonstration and is not the clustering target.

### Context

- `client_id`
- `content_id`
- March and April row counts
- availability-day counts

### Excluded

- `future_ctr_pct` from the honest features: it belongs to the outcome month.
- `future_high_ctr_leak`: an intentional copy of the proxy used once to demonstrate leakage, then deleted.
- June 2026 rows: sealed test month.
- identifiers: grouping and joining only.


## 3. Three verification queries

There are exactly three contract-verification queries below:

1. grain;
2. slice row count and date span;
3. availability using `IS TRUE`.


### Verification query 1 — grain

The documented raw grain is one row per `report_date × client_id × content_id`. An empty result means no duplicate raw-grain combinations were found in March.


In [ ]:
grain_query = f"""
SELECT
    report_date,
    client_id,
    content_id,
    COUNT(*) AS rows_at_grain
FROM read_parquet('{MARCH}')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 10
"""

grain_result = con.sql(grain_query).df()
display(grain_result)

assert grain_result.empty, (
    "The expected daily grain did not hold. Inspect duplicates before continuing."
)
print("PASS: no duplicate report_date × client_id × content_id combinations found.")


### Verification query 2 — row count and date span

This proves how many raw daily rows are in the March slice and confirms that the partition covers the intended dates.


In [ ]:
count_span_query = f"""
SELECT
    COUNT(*) AS raw_rows,
    COUNT(DISTINCT client_id) AS clients,
    COUNT(DISTINCT content_id) AS content_items,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{MARCH}')
"""

count_span_result = con.sql(count_span_query).df()
display(count_span_result)

assert str(count_span_result.loc[0, "min_report_date"]).startswith("2026-03-01")
assert str(count_span_result.loc[0, "max_report_date"]).startswith("2026-03-31")
print("PASS: the partition date span matches March 2026.")


### Verification query 3 — GA4 availability

Rows before a client's analytics history begins can contain zero-filled GA4 measures. The safe check is an explicit `ga4_data_available IS TRUE` filter, not `sessions > 0`.


In [ ]:
availability_query = f"""
SELECT
    COUNT(*) AS all_march_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
        / NULLIF(COUNT(*), 0),
        2
    ) AS ga4_available_pct
FROM read_parquet('{MARCH}')
"""

availability_result = con.sql(availability_query).df()
display(availability_result)

surviving = int(availability_result.loc[0, "ga4_available_rows"])
assert surviving > 0, "No GA4-available rows survived; check access or schema."
print(f"PASS: {surviving:,} March rows survive ga4_data_available IS TRUE.")


## 4. Build the five-feature frame

Each feature uses only information available by the end of March 2026.

- **`log_impressions` — available when?** Knowable at the decision moment because it is calculated only from impressions recorded during March.
- **`ctr_pct` — available when?** Knowable because March clicks and impressions are complete before the April outcome window begins.
- **`avg_position` — available when?** Knowable because it uses only March GSC positions, weighted by March impressions.
- **`active_days` — available when?** Knowable because it counts March days with measured search visibility.
- **`engagement_rate_pct` — available when?** Knowable because it uses March GA4 sessions only on rows explicitly marked available.

The April columns are retained only to build the temporary leakage-demo proxy. They are not honest March features.


In [ ]:
feature_query = f"""
WITH march AS (
    SELECT
        client_id,
        content_id,
        SUM(COALESCE(gsc_impressions, 0)) AS impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks,
        SUM(
            CASE
                WHEN gsc_impressions > 0 AND gsc_avg_position > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        ) AS weighted_position_sum,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days,
        SUM(
            CASE WHEN ga4_data_available IS TRUE
                 THEN COALESCE(ga4_sessions, 0) ELSE 0 END
        ) AS ga4_sessions_available,
        SUM(
            CASE WHEN ga4_data_available IS TRUE
                 THEN COALESCE(ga4_engaged_sessions, 0) ELSE 0 END
        ) AS ga4_engaged_sessions_available,
        COUNT(DISTINCT CASE WHEN ga4_data_available IS TRUE THEN report_date END)
            AS ga4_available_days
    FROM read_parquet('{MARCH}')
    GROUP BY 1, 2
),
april AS (
    SELECT
        client_id,
        content_id,
        SUM(COALESCE(gsc_impressions, 0)) AS future_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS future_clicks
    FROM read_parquet('{APRIL}')
    GROUP BY 1, 2
)
SELECT
    m.client_id,
    m.content_id,

    -- Five honest March features
    LN(1 + m.impressions) AS log_impressions,
    100.0 * m.clicks / NULLIF(m.impressions, 0) AS ctr_pct,
    m.weighted_position_sum / NULLIF(m.impressions, 0) AS avg_position,
    m.active_days,
    100.0 * m.ga4_engaged_sessions_available
        / NULLIF(m.ga4_sessions_available, 0) AS engagement_rate_pct,

    -- Context and future proxy ingredients, never honest features
    m.impressions AS march_impressions,
    m.ga4_available_days,
    a.future_impressions,
    a.future_clicks,
    100.0 * a.future_clicks / NULLIF(a.future_impressions, 0) AS future_ctr_pct
FROM march m
INNER JOIN april a USING (client_id, content_id)
WHERE
    m.impressions > 0
    AND m.ga4_available_days > 0
    AND m.ga4_sessions_available > 0
    AND a.future_impressions >= 20
"""

frame = con.sql(feature_query).df()

feature_cols = [
    "log_impressions",
    "ctr_pct",
    "avg_position",
    "active_days",
    "engagement_rate_pct",
]

assert len(feature_cols) == 5
assert set(feature_cols).issubset(frame.columns)
assert len(frame) > 100, "Too few eligible pages for the quick leakage demonstration."

display(frame.head())
print("Feature-frame shape:", frame.shape)
print("\nMissing values in the five features:")
display(frame[feature_cols].isna().sum().to_frame("missing"))


## 5. Deliberate leakage experiment

This experiment is intentionally supervised even though the capstone is clustering. Its only purpose is to make leakage visible.

### Honest setup

At the end of March, the model may use only the five March features. The temporary proxy is based on April CTR and is therefore future information.

### Leaked setup

I add `future_high_ctr_leak`, which is an exact copy of the April-derived proxy. This is information the model could not know at the March decision moment. The score should jump toward perfect for the wrong reason.

After showing the jump, the leaked column is deleted and the honest score is kept.


In [ ]:
# Create a balanced, temporary April proxy among eligible pages.
future_ctr_median = frame["future_ctr_pct"].median()
frame["future_high_ctr"] = (frame["future_ctr_pct"] > future_ctr_median).astype(int)

X_honest = frame[feature_cols].copy()
y = frame["future_high_ctr"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

honest_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)
honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)
honest_prob = honest_model.predict_proba(X_test)[:, 1]

honest_accuracy = accuracy_score(y_test, honest_pred)
honest_balanced_accuracy = balanced_accuracy_score(y_test, honest_pred)
honest_auc = roc_auc_score(y_test, honest_prob)

print("HONEST MODEL — March features only")
print(f"Accuracy:          {honest_accuracy:.3f}")
print(f"Balanced accuracy: {honest_balanced_accuracy:.3f}")
print(f"ROC AUC:           {honest_auc:.3f}")


In [ ]:
# Add one label-derived column on purpose.
frame["future_high_ctr_leak"] = frame["future_high_ctr"]

leaked_cols = feature_cols + ["future_high_ctr_leak"]
X_leaked = frame[leaked_cols].copy()

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaked,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

leaked_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)
leaked_model.fit(Xl_train, yl_train)

leaked_pred = leaked_model.predict(Xl_test)
leaked_prob = leaked_model.predict_proba(Xl_test)[:, 1]

leaked_accuracy = accuracy_score(yl_test, leaked_pred)
leaked_balanced_accuracy = balanced_accuracy_score(yl_test, leaked_pred)
leaked_auc = roc_auc_score(yl_test, leaked_prob)

comparison = pd.DataFrame(
    {
        "setup": ["Honest: March features", "Leaked: label copy included"],
        "accuracy": [honest_accuracy, leaked_accuracy],
        "balanced_accuracy": [honest_balanced_accuracy, leaked_balanced_accuracy],
        "roc_auc": [honest_auc, leaked_auc],
    }
)
display(comparison.round(3))

print(
    "Leakage lesson: the leaked model looks excellent because it was given "
    "the answer, not because it learned a generalizable relationship."
)

# Remove the leaked column and assert it cannot be used later.
frame.drop(columns=["future_high_ctr_leak"], inplace=True)
assert "future_high_ctr_leak" not in frame.columns
assert "future_high_ctr_leak" not in feature_cols

print("\nLeaked column deleted. The honest ROC AUC retained:", round(honest_auc, 3))


## 6. Named limitation

This slice cannot tell me why a page performed in a certain way or whether a content change caused an outcome. It only contains measured search and analytics signals.

A second limitation is the unbalanced panel: clients begin contributing GSC and GA4 history on different dates. Even within a calendar month, data coverage can differ. I therefore check explicit availability flags and treat the output as directional decision support, not a causal claim or a model of Google's algorithm.


In [ ]:
# Compact final audit table for the notebook.
audit = pd.DataFrame(
    {
        "contract_item": [
            "Raw grain verified",
            "March date span verified",
            "GA4 availability checked with IS TRUE",
            "Exactly five honest features",
            "Future label-derived leak removed",
            "June sealed",
        ],
        "status": [
            grain_result.empty,
            str(count_span_result.loc[0, "min_report_date"]).startswith("2026-03-01")
            and str(count_span_result.loc[0, "max_report_date"]).startswith("2026-03-31"),
            int(availability_result.loc[0, "ga4_available_rows"]) > 0,
            len(feature_cols) == 5,
            "future_high_ctr_leak" not in frame.columns,
            True,
        ],
    }
)
display(audit)
assert audit["status"].all()
print("All executable contract checks passed.")


## 7. Self-check

Before submission:

- [x] Five plain-word contract answers are included.
- [x] Exactly three verification queries are included.
- [x] Grain, row count/date span, and availability are verified.
- [x] Availability uses `ga4_data_available IS TRUE`.
- [x] The feature frame contains exactly five honest features.
- [x] Every feature has an “available when?” explanation.
- [x] One label-derived column is deliberately added, scored, and removed.
- [x] The honest score is retained.
- [x] One limitation is named.
- [x] No token, client name, URL, title, domain, or raw query is printed.
- [ ] Run all cells in Colab and confirm outputs are visible.
- [ ] Save to `work/notebooks/w03_data_contract.ipynb`.
- [ ] Commit to the public repo and submit the repo URL.
